# Item-based collaborative filtering

This notebook evaluates a personalized recommender using overlap in users' positive movie feedback. Its purpose is to test whether behavior-derived item similarities retrieve more later likes than the popularity baseline from notebook 01.

Run notebook 01 first. We load its saved **per-user chronological 60/20/20** split rather than independently recreating partition boundaries. Only training interactions build similarities, user profiles, and the candidate catalog. Validation chooses among models; the test partition remains reserved for final evaluation after model selection.

A rating of at least 4.0 is positive feedback. Every model uses the same training catalog, excludes all movies already rated by that user, and is evaluated on the same users with positive validation feedback. All positive validation movies remain in the Recall@10 denominator, including movies outside the training catalog.

## Load the canonical split and evaluation contract

The manifest verifies raw-data and split-file fingerprints. Keeping artifacts in a split-specific directory prevents accidentally comparing refreshed recommendations with result files from the former global date split. The shared loader also provides the provenance-aware result writer used at the end of this notebook.

In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd

SPLIT_ID = "per_user_chronological_60_20_20_v1"
POSITIVE_THRESHOLD = 4.0
TOP_K = 10

# Resolve the project from either its root or any nested notebook directory.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "raw" / "ml-latest-small").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run from the project containing data/raw/ml-latest-small.")

raw_dir = data_dir = project_root / "data" / "raw" / "ml-latest-small"
processed_dir = project_root / "data" / "processed" / SPLIT_ID
manifest_path = processed_dir / "split_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Run notebook 01 first to write the canonical split.")
split_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert split_manifest["split_id"] == SPLIT_ID
assert split_manifest["positive_threshold"] == POSITIVE_THRESHOLD
assert split_manifest["top_k"] == TOP_K

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename, expected_hash in split_manifest["raw_sha256"].items():
    if file_sha256(raw_dir / filename) != expected_hash:
        raise ValueError(f"{filename} changed; regenerate the split in notebook 01.")

movies = pd.read_csv(raw_dir / "movies.csv")
loaded_splits = {}
for name in ("train", "validation"):
    path = processed_dir / f"{name}_ratings.csv"
    expected = split_manifest["files"][path.name]
    if file_sha256(path) != expected["sha256"]:
        raise ValueError(f"{path.name} does not match the split manifest.")
    loaded_splits[name] = pd.read_csv(path, parse_dates=["rated_at"])
    assert len(loaded_splits[name]) == expected["rows"]

train_ratings = loaded_splits["train"]
validation_ratings = loaded_splits["validation"]

def save_validation_results(results, filename):
    """Attach the split and evaluation contract to each per-user result row."""
    saved = results.copy()
    saved["split_id"] = SPLIT_ID
    saved["split_fingerprint"] = split_manifest["split_fingerprint"]
    saved["evaluation_split"] = "validation"
    saved["positive_threshold"] = POSITIVE_THRESHOLD
    saved["k"] = TOP_K
    saved.to_csv(processed_dir / filename, index=False)

def validate_result_provenance(results, label="results"):
    expected = {
        "split_id": SPLIT_ID,
        "split_fingerprint": split_manifest["split_fingerprint"],
        "evaluation_split": "validation",
        "positive_threshold": POSITIVE_THRESHOLD,
        "k": TOP_K,
    }
    for column, value in expected.items():
        assert column in results and results[column].eq(value).all(), (
            f"{label}: missing or incompatible {column}; rerun its source notebook."
        )

print("Split:", SPLIT_ID)
print("Training ratings:", len(train_ratings))
print("Validation ratings:", len(validation_ratings))


## Represent positive feedback

The user-by-movie matrix contains one for a training rating at or above the positive threshold and zero otherwise. Here, an unrated movie and a lower rating both contribute no positive evidence; we are modeling positive co-occurrence rather than predicting stars. All training movies remain columns, including movies with no positive feedback.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

training_feedback = train_ratings[["userId", "movieId", "rating"]].copy()
training_feedback["liked"] = (training_feedback["rating"] >= POSITIVE_THRESHOLD).astype(int)

user_movie_matrix = (
    training_feedback.pivot(index="userId", columns="movieId", values="liked")
    .fillna(0)
    .astype(int)
)
movie_user_matrix = user_movie_matrix.T

assert user_movie_matrix.index.is_unique
assert movie_user_matrix.index.is_unique
print("Users:", user_movie_matrix.shape[0])
print("Training movies:", user_movie_matrix.shape[1])
print("Positive training ratings:", training_feedback["liked"].sum())
user_movie_matrix.iloc[:5, :8]


## Construct the shared candidate ranking

Candidates are exactly the movies rated somewhere in training. Their training rating counts serve as the popularity fallback and as a deterministic secondary sort key when similarity scores tie. Lower movie ID resolves any remaining ties. Validation ratings contribute neither candidates nor popularity counts.

In [ ]:
movie_popularity = (
    train_ratings.groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(movies[["movieId", "title"]], on="movieId", how="left", validate="one_to_one")
    .sort_values(["training_rating_count", "movieId"], ascending=[False, True])
    .reset_index(drop=True)
)

assert set(movie_popularity["movieId"]) == set(train_ratings["movieId"])
movie_popularity.head(TOP_K)


## Inspect cosine similarity between two movies

For binary feedback, movie similarity equals the count of users who liked both movies divided by the geometric mean of their like counts:

$$\operatorname{sim}(A,B)=\frac{|L_A\cap L_B|}{\sqrt{|L_A|\,|L_B|}}.$$

We demonstrate this with Toy Story and Jumanji when available. A movie with no training likes has a zero vector; its similarity is defined here as zero, matching scikit-learn's zero-vector behavior.

In [ ]:
seed_a = 1 if 1 in user_movie_matrix.columns else int(user_movie_matrix.columns[0])
seed_b = 2 if 2 in user_movie_matrix.columns else int(user_movie_matrix.columns[-1])
likes_a = user_movie_matrix[seed_a]
likes_b = user_movie_matrix[seed_b]

shared_likes = int((likes_a * likes_b).sum())
denominator = np.sqrt(float(likes_a.sum()) * float(likes_b.sum()))
manual_similarity = shared_likes / denominator if denominator > 0 else 0.0
library_similarity = cosine_similarity(
    movie_user_matrix.loc[[seed_a]], movie_user_matrix.loc[[seed_b]]
)[0, 0]

assert np.isclose(manual_similarity, library_similarity)
print("Movie IDs:", seed_a, seed_b)
print("Positive feedback counts:", int(likes_a.sum()), int(likes_b.sum()))
print("Shared positive users:", shared_likes)
print("Cosine similarity:", library_similarity)


## Inspect a seed movie's nearest neighbors

This diagnostic compares one movie with the entire training catalog. The seed is removed from its own neighbor list. These item neighbors explain the representation; user recommendations below additionally combine multiple liked movies and remove the user's full rated history.

In [ ]:
seed_scores = cosine_similarity(movie_user_matrix.loc[[seed_a]], movie_user_matrix)[0]
similar_movies = pd.DataFrame({
    "movieId": movie_user_matrix.index,
    "similarity": seed_scores,
}).merge(movie_popularity, on="movieId", how="left", validate="one_to_one")

similar_movies = (
    similar_movies.loc[similar_movies["movieId"] != seed_a]
    .sort_values(
        ["similarity", "training_rating_count", "movieId"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)
similar_movies[["movieId", "title", "similarity"]].head(TOP_K)


## Build a user's item-similarity scores

Each candidate receives its mean cosine similarity to the movies this user liked during training. All likes receive equal weight. The example explicitly handles an empty liked history with zero scores so it follows the same popularity fallback as the reusable recommender.

In [ ]:
example_user_id = 1 if 1 in set(train_ratings["userId"]) else int(train_ratings["userId"].iloc[0])
user_history = train_ratings.loc[train_ratings["userId"] == example_user_id]
liked_ids = user_history.loc[user_history["rating"] >= POSITIVE_THRESHOLD, "movieId"]

if liked_ids.empty:
    mean_scores = np.zeros(len(movie_user_matrix), dtype=float)
else:
    similarities_to_liked = cosine_similarity(movie_user_matrix.loc[liked_ids], movie_user_matrix)
    mean_scores = similarities_to_liked.mean(axis=0)

example_scores = pd.Series(mean_scores, index=movie_user_matrix.index)
print("Example user:", example_user_id)
print("Training ratings:", len(user_history))
print("Positive training ratings:", len(liked_ids))


## Exclude seen titles and rank candidates

Every rated training movie is excluded, including movies the user disliked. Scores rank descending; training popularity and movie ID break ties. This is a retrieval score, not a predicted star rating or a calibrated probability.

In [ ]:
example_recommendations = movie_popularity.loc[
    ~movie_popularity["movieId"].isin(user_history["movieId"])
].copy()
example_recommendations["score"] = example_recommendations["movieId"].map(example_scores).fillna(0.0)
example_recommendations = (
    example_recommendations.sort_values(
        ["score", "training_rating_count", "movieId"], ascending=[False, False, True]
    )
    .head(TOP_K)
    .reset_index(drop=True)
)
example_recommendations[["movieId", "title", "score", "training_rating_count"]]


## Define the recommender

The function uses the same calculation and tie rules as the walkthrough. A user without usable positive history receives the popularity order because all candidate scores remain zero. Fallback users stay in the evaluation cohort; dropping them would change which users are compared across models.

In [ ]:
def recommend_item_based(
    user_id,
    training_data,
    movie_user_matrix,
    popularity_ranking,
    n=10,
):
    """Rank unseen fitting-catalog movies by mean similarity to training likes."""
    history = training_data.loc[training_data["userId"] == user_id]
    liked_ids = history.loc[
        (history["rating"] >= POSITIVE_THRESHOLD)
        & history["movieId"].isin(movie_user_matrix.index),
        "movieId",
    ]

    recommendations = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(history["movieId"])
    ].copy()
    recommendations["score"] = 0.0

    if not liked_ids.empty:
        similarities = cosine_similarity(
            movie_user_matrix.loc[liked_ids], movie_user_matrix
        )
        scores = pd.Series(similarities.mean(axis=0), index=movie_user_matrix.index)
        recommendations["score"] = recommendations["movieId"].map(scores).fillna(0.0)

    return (
        recommendations.sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True],
        )
        .head(n)
        .reset_index(drop=True)
    )


## Check representative recommendations

Inspecting example outputs helps catch row alignment and exclusion errors before aggregate evaluation. The walkthrough and reusable function should return the same ranked movie IDs for the example user.

In [ ]:
function_example = recommend_item_based(
    example_user_id, train_ratings, movie_user_matrix, movie_popularity, n=TOP_K
)
assert function_example["movieId"].tolist() == example_recommendations["movieId"].tolist()

for user_id in sorted(train_ratings["userId"].unique())[:2]:
    print("Recommendations for user:", user_id)
    display(recommend_item_based(user_id, train_ratings, movie_user_matrix, movie_popularity, n=TOP_K))


## Evaluate Recall@10 on validation

For each user with at least one positive validation rating, recall is the number of retrieved positive validation movies divided by that user's total positive validation movies. The final mean gives each eligible user equal weight. Users without positive validation feedback have no defined recall denominator and are reported separately.

The denominator includes positives outside the training candidate catalog. Reporting how many positives are in that catalog makes the retrieval ceiling visible. Assertions check uniqueness, candidate membership, rated-movie exclusions, finite scores, and the expected recommendation-list length.

In [ ]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= POSITIVE_THRESHOLD
].copy()

training_users = set(train_ratings["userId"])
liked_training_users = set(
    train_ratings.loc[train_ratings["rating"] >= POSITIVE_THRESHOLD, "userId"]
)
candidate_ids = set(movie_popularity["movieId"])
evaluation_rows = []

for user_id, positives in validation_positives.groupby("userId", sort=True):
    positive_ids = set(positives["movieId"])
    seen_ids = set(train_ratings.loc[train_ratings["userId"] == user_id, "movieId"])
    recommendations = recommend_item_based(
        user_id=user_id,
        training_data=train_ratings,
        movie_user_matrix=movie_user_matrix,
        popularity_ranking=movie_popularity,
        n=TOP_K,
    )
    recommended_ids = set(recommendations["movieId"])

    assert recommendations["movieId"].is_unique
    assert len(recommendations) == min(TOP_K, len(candidate_ids - seen_ids))
    assert recommended_ids.issubset(candidate_ids)
    assert recommended_ids.isdisjoint(seen_ids)
    assert np.isfinite(recommendations["score"]).all()

    hits = len(positive_ids & recommended_ids)
    evaluation_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_ids),
        "positive_movies_in_training_catalog": len(positive_ids & candidate_ids),
        "recommended_movie_count": len(recommendations),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_ids),
        "has_training_history": user_id in training_users,
        "has_liked_training_history": user_id in liked_training_users,
    })

item_based_results = pd.DataFrame(evaluation_rows)
assert not item_based_results.empty, "No users have positive validation ratings."
assert item_based_results["userId"].is_unique
assert item_based_results["has_training_history"].all()

print("Users evaluated:", len(item_based_results))
print("Users with no positive validation ratings:",
      validation_ratings["userId"].nunique() - len(item_based_results))
print("Mean Recall@10:", item_based_results["recall_at_10"].mean())
item_based_results.head()


## Describe personalization coverage

The revised split gives every evaluated user training history. Some users may still have no training ratings at or above the positive threshold; they receive the documented fallback. Grouping by positive-history availability is therefore more informative than the former new-user versus returning-user comparison.

In [ ]:
personalization_summary = (
    item_based_results.groupby("has_liked_training_history")
    .agg(users=("userId", "size"), mean_recall_at_10=("recall_at_10", "mean"),
         total_hits_at_10=("hits_at_10", "sum"))
    .reset_index()
)
personalization_summary


## Compare users with the popularity baseline

We reload notebook 01's validation results and verify their split provenance before joining. An outer join and denominator check prevent silently comparing different users or different sets of relevant movies. The paired differences show how many people improve, worsen, or remain unchanged; an aggregate mean alone can hide that variation.

In [ ]:
popularity_results = pd.read_csv(processed_dir / "popularity_validation_results.csv")
validate_result_provenance(popularity_results)

paired_results = item_based_results.merge(
    popularity_results[["userId", "positive_movie_count", "recall_at_10"]],
    on="userId",
    how="outer",
    suffixes=("_model", "_popularity"),
    indicator=True,
    validate="one_to_one",
)
assert paired_results["_merge"].eq("both").all()
assert (
    paired_results["positive_movie_count_model"]
    == paired_results["positive_movie_count_popularity"]
).all()

paired_results["recall_change"] = (
    paired_results["recall_at_10_model"] - paired_results["recall_at_10_popularity"]
)

validation_comparison = pd.DataFrame({
    "model": ["Popularity", "Item-based CF"],
    "users": [len(paired_results), len(paired_results)],
    "mean_recall_at_10": [
        paired_results["recall_at_10_popularity"].mean(),
        paired_results["recall_at_10_model"].mean(),
    ],
})
display(validation_comparison)

change = paired_results["recall_change"]
print("Users with improved recall:", change.gt(0).sum())
print("Users with worse recall:", change.lt(0).sum())
print("Users with unchanged recall:", change.eq(0).sum())
print("Mean gain in percentage points:", 100 * change.mean())


## Save validation results

The result writer attaches the split ID, split fingerprint, relevance threshold, and recommendation-list size. Notebook 06 can then reject stale or incompatible evaluations instead of mixing them into the model comparison.

In [ ]:
save_validation_results(item_based_results, "item_based_validation_results.csv")
print("Saved collaborative-filtering validation results.")


## Interpretation and limitations

Read the freshly computed comparison and paired differences before concluding that item similarity improves retrieval. Scores and conclusions from the former global calendar split do not apply to this cohort.

This model treats every positive rating equally, averages over potentially varied preferences, and cannot recommend movies absent from training. Popularity tie-breaking can influence rankings when evidence is sparse. Per-user chronology preserves each user's rating order but does not enforce one global deployment date: another user's training rating may occur after this user's held-out rating. This is an existing-user offline benchmark rather than a historical production replay.

Model selection uses validation results only. The held-out test partition must remain unused until the selected model and settings are fixed.